# CIC-IDS-2017 - Data Cleaning

This notebook iterates through all CSV files in the TrafficLabelling directory, applies cleaning steps, and saves the cleaned files.

In [9]:
import pandas as pd
import numpy as np
import os
import dotenv
import warnings

warnings.filterwarnings('ignore')
dotenv.load_dotenv()

True

In [10]:
data_dir = os.getenv('DATA_DIR', './data')
traffic_dir = os.path.join(data_dir, 'GeneratedLabelledFlows', 'TrafficLabelling')

csv_files = sorted([f for f in os.listdir(traffic_dir) if f.endswith('.csv') and '_cleaned' not in f])
print(f'Found {len(csv_files)} CSV files in {traffic_dir}:\n')
for f in csv_files:
    print(f'  {f}')

Found 8 CSV files in /mnt/data/capstone/GeneratedLabelledFlows/TrafficLabelling:

  Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
  Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
  Friday-WorkingHours-Morning.pcap_ISCX.csv
  Monday-WorkingHours.pcap_ISCX.csv
  Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
  Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
  Tuesday-WorkingHours.pcap_ISCX.csv
  Wednesday-workingHours.pcap_ISCX.csv


In [11]:
id_cols = ['Flow ID', 'Timestamp']

correlated_drops = [
    'Total Backward Packets',
    'Subflow Fwd Packets',
    'Subflow Bwd Packets',
    'Total Length of Bwd Packets',
    'Subflow Bwd Bytes',
    'Subflow Fwd Bytes',
    'Avg Fwd Segment Size',
    'Avg Bwd Segment Size',
    'Fwd Header Length.1',
    'Fwd PSH Flags',
    'CWE Flag Count',
    'ECE Flag Count',
    'Average Packet Size',
    'Fwd IAT Total',
    'Fwd IAT Max',
    'Idle Max',
    'Idle Min',
]

def clean(df):
    rows_before = len(df)
    non_id_cols = [c for c in df.columns if c not in id_cols]
    df = df.dropna(subset=non_id_cols, how='all')
    rows_after = len(df)
    print(f'  Removed {rows_before - rows_after:,} entirely-null rows (ignoring ID columns)')

    cols_before = df.shape[1]
    drops = [c for c in correlated_drops if c in df.columns]
    df = df.drop(columns=drops)
    print(f'  Dropped {len(drops)} near-perfectly correlated columns ({cols_before} -> {df.shape[1]})')

    return df

In [12]:
for filename in csv_files:
    filepath = os.path.join(traffic_dir, filename)
    df = pd.read_csv(filepath, encoding='cp1252')
    df.columns = df.columns.str.strip()
    print(f'{filename}: {df.shape[0]:,} rows')

    df = clean(df)

    out_name = filename.replace('.csv', '_cleaned.csv')
    out_path = os.path.join(traffic_dir, out_name)
    df.to_csv(out_path, index=False)
    print(f'  Saved {df.shape[0]:,} rows to {out_name}\n')

Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv: 225,745 rows
  Removed 0 entirely-null rows (ignoring ID columns)
  Dropped 17 near-perfectly correlated columns (85 -> 68)
  Saved 225,745 rows to Friday-WorkingHours-Afternoon-DDos.pcap_ISCX_cleaned.csv

Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv: 286,467 rows
  Removed 0 entirely-null rows (ignoring ID columns)
  Dropped 17 near-perfectly correlated columns (85 -> 68)
  Saved 286,467 rows to Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX_cleaned.csv

Friday-WorkingHours-Morning.pcap_ISCX.csv: 191,033 rows
  Removed 0 entirely-null rows (ignoring ID columns)
  Dropped 17 near-perfectly correlated columns (85 -> 68)
  Saved 191,033 rows to Friday-WorkingHours-Morning.pcap_ISCX_cleaned.csv

Monday-WorkingHours.pcap_ISCX.csv: 529,918 rows
  Removed 0 entirely-null rows (ignoring ID columns)
  Dropped 17 near-perfectly correlated columns (85 -> 68)
  Saved 529,918 rows to Monday-WorkingHours.pcap_ISCX_cleaned.csv

Thursday-Wor